In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split

Matplotlib is building the font cache; this may take a moment.


In [2]:
df = pd.read_csv("../dataset/processed/cleaned_stroke_dataset.csv")

df.head()

,age,hypertension,heart_disease,avg_glucose_level,bmi,stroke,gender_Female,gender_Male,gender_Other,ever_married_No,...,work_type_Never_worked,work_type_Private,work_type_Self-employed,work_type_children,Residence_type_Rural,Residence_type_Urban,smoking_status_Unknown,smoking_status_formerly smoked,smoking_status_never smoked,smoking_status_smokes
0,67.0,0,1,228.69,36.6,1,0,1,0,0,...,0,1,0,0,0,1,0,1,0,0
1,61.0,0,0,202.21,28.1,1,1,0,0,0,...,0,0,1,0,1,0,0,0,1,0
2,80.0,0,1,105.92,32.5,1,0,1,0,0,...,0,1,0,0,1,0,0,0,1,0
3,49.0,0,0,171.23,34.4,1,1,0,0,0,...,0,1,0,0,0,1,0,0,0,1
4,79.0,1,0,174.12,24.0,1,1,0,0,0,...,0,0,1,0,1,0,0,0,1,0


In [3]:
print("Dataset shape:", df.shape)

print("\nMissing values:")
print(df.isnull().sum().sum())

print("\nDuplicate rows:", df.duplicated().sum())

print("\nTarget distribution:")
print(df["stroke"].value_counts())

Dataset shape: (5110, 22)

Missing values:
0

Duplicate rows: 0

Target distribution:
stroke
0    4861
1     249
Name: count, dtype: int64


In [4]:
X = df.drop("stroke", axis=1)
y = df["stroke"]

print("Features (X):", X.shape)
print("Target (y):", y.shape)

Features (X): (5110, 21)
Target (y): (5110,)


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts())

print("\nTesting target distribution:")
print(y_test.value_counts())

Training data: (4088, 21)
Testing data: (1022, 21)

Training target distribution:
stroke
0    3889
1     199
Name: count, dtype: int64

Testing target distribution:
stroke
0    972
1     50
Name: count, dtype: int64


In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

print("Training data is ready.")
print("Stroke cases:", y_train.sum())
print("Non-stroke cases:", (y_train == 0).sum())

Training data is ready.
Stroke cases: 199
Non-stroke cases: 3889


In [8]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

logistic_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        class_weight="balanced",
        max_iter=2000,
        random_state=42
    ))
])

logistic_model.fit(X_train, y_train)

print("Logistic Regression training completed successfully!")

Logistic Regression training completed successfully!


In [9]:
from sklearn.tree import DecisionTreeClassifier

decision_tree_model = DecisionTreeClassifier(
    class_weight="balanced",
    random_state=42
)

decision_tree_model.fit(X_train, y_train)

print("Decision Tree training completed successfully!")

Decision Tree training completed successfully!


In [10]:
from sklearn.ensemble import RandomForestClassifier

random_forest_model = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

random_forest_model.fit(X_train, y_train)

print("Random Forest training completed successfully!")

Random Forest training completed successfully!


In [11]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

models = {
    "Logistic Regression": logistic_model,
    "Decision Tree": decision_tree_model,
    "Random Forest": random_forest_model
}

results = []

for name, model in models.items():
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1 Score": f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, y_prob)
    })

comparison_df = pd.DataFrame(results)

comparison_df

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,Logistic Regression,0.745597,0.137931,0.80,0.235294,0.843765
1,Decision Tree,0.932485,0.193548,0.12,0.148148,0.547140
2,Random Forest,0.937378,0.208333,0.10,0.135135,0.782006


In [12]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "classifier__C": [0.01, 0.1, 1, 10, 100],
    "classifier__class_weight": ["balanced", None],
    "classifier__solver": ["liblinear", "lbfgs"]
}

grid_search = GridSearchCV(
    logistic_model,
    param_grid,
    cv=5,
    scoring="roc_auc",
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("Best parameters:")
print(grid_search.best_params_)

print("\nBest cross-validation ROC-AUC:")
print(grid_search.best_score_)

Best parameters:
{'classifier__C': 0.1, 'classifier__class_weight': 'balanced', 'classifier__solver': 'lbfgs'}

Best cross-validation ROC-AUC:
0.8375837683356965


In [13]:
best_logistic_model = grid_search.best_estimator_

y_pred_tuned = best_logistic_model.predict(X_test)
y_prob_tuned = best_logistic_model.predict_proba(X_test)[:, 1]

print("Tuned Logistic Regression Results")
print("----------------------------------")
print("Accuracy :", accuracy_score(y_test, y_pred_tuned))
print("Precision:", precision_score(y_test, y_pred_tuned, zero_division=0))
print("Recall   :", recall_score(y_test, y_pred_tuned, zero_division=0))
print("F1 Score :", f1_score(y_test, y_pred_tuned, zero_division=0))
print("ROC-AUC  :", roc_auc_score(y_test, y_prob_tuned))

Tuned Logistic Regression Results
----------------------------------
Accuracy : 0.7455968688845401
Precision: 0.13793103448275862
Recall   : 0.8
F1 Score : 0.23529411764705882
ROC-AUC  : 0.8431069958847737


In [14]:
import joblib

final_model = logistic_model

joblib.dump(final_model, "../models/stroke_model.pkl")

print("Final model saved successfully!")

Final model saved successfully!


In [15]:
print("Final model features:")
print(list(X.columns))

Final model features:
['age', 'hypertension', 'heart_disease', 'avg_glucose_level', 'bmi', 'gender_Female', 'gender_Male', 'gender_Other', 'ever_married_No', 'ever_married_Yes', 'work_type_Govt_job', 'work_type_Never_worked', 'work_type_Private', 'work_type_Self-employed', 'work_type_children', 'Residence_type_Rural', 'Residence_type_Urban', 'smoking_status_Unknown', 'smoking_status_formerly smoked', 'smoking_status_never smoked', 'smoking_status_smokes']
